# COMP663 Assignment 2 — Task 4: Neural Architecture Search

This notebook runs Task 4 only. It uses the same stratified 60/20/20 split, training-only scaler, seed, validation metric, and 20-epoch trial budget as Tasks 2 and 3.

### 4.1 Hyperparameters to optimise

The NAS searches the number of hidden layers, common hidden-layer width, and activation function. It also searches learning rate, batch size, and weight decay from Task 1.

### 4.2 Objective, NAS search space, and search strategy

The objective is to maximise validation macro-F1. Each candidate genome contains hidden layers {1, 2, 3}, width {16, 24, 32, 48, 64}, activation {Sigmoid, ReLU}, learning rate {0.0001, 0.001, 0.01}, batch size {256, 512, 1024}, and weight decay {0, 0.00001, 0.0001, 0.001}. I use evolutionary NAS from Tutorial 5: each generation retains the best candidate, creates children through tournament selection and crossover, and randomly mutates genes.

### 4.3 Evaluation procedure and primary metric

Every candidate trains on the same 60% training split. Macro-F1 and balanced accuracy are measured only on the same 20% validation split. The scaler is fitted only on the training split.

### 4.4 Computational budget and justification

The budget is 12 candidates: a population of four across three generations, with 20 epochs per candidate. This is the same trial and epoch budget as Tasks 2 and 3, while remaining practical on a Kaggle T4 GPU.

### 4.5 Apply evolutionary NAS


In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
SEARCH_EPOCHS = 20
POPULATION_SIZE = 4
GENERATIONS = 3
NAS_TRIALS = POPULATION_SIZE * GENERATIONS

if Path("/kaggle").exists() and not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU was not allocated. Enable a T4 GPU before running.")
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    RUNTIME_DEVICE = f"{torch.cuda.get_device_name(0)} x{torch.cuda.device_count()}"
elif DEVICE.type == "mps":
    RUNTIME_DEVICE = "Apple Silicon MPS"
else:
    RUNTIME_DEVICE = "CPU"

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

def format_decimal(value):
    return np.format_float_positional(float(value), unique=True, trim="-")

def log_cell(name, started, configuration):
    elapsed = format_decimal(time.perf_counter() - started)
    print(f"{name}: device={RUNTIME_DEVICE}; configuration={configuration}; elapsed_seconds={elapsed}")

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
KAGGLE_DATA_PATH = Path("/kaggle/input/datasets/yangliunz/comp663-a2-forest-cove/forest_cover_data.csv")
kaggle_data_paths = ([KAGGLE_DATA_PATH] if KAGGLE_DATA_PATH.exists() else []) + list(Path("/kaggle/input").rglob("forest_cover_data.csv"))
DATA_PATH = kaggle_data_paths[0] if kaggle_data_paths else ROOT / "data" / "forest_cover_data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"forest_cover_data.csv not found: {DATA_PATH}")

data_started = time.perf_counter()
data = pd.read_csv(DATA_PATH).dropna(subset=["Cover_Type"])
target = "Cover_Type"
feature_names = [column for column in data.columns if column != target]
continuous_features = [column for column in feature_names if not column.startswith("Wilderness_Area")]
assert data.shape == (571_012, 15), data.shape

train_validation_frame, test_frame = train_test_split(data, test_size=0.20, stratify=data[target], random_state=SEED)
train_frame, validation_frame = train_test_split(train_validation_frame, test_size=0.25, stratify=train_validation_frame[target], random_state=SEED)
scaler = StandardScaler().fit(train_frame[continuous_features])

def prepare_data(frame):
    features = frame[feature_names].astype("float32").copy()
    features[continuous_features] = scaler.transform(features[continuous_features])
    return features.to_numpy(), frame[target].to_numpy(dtype=np.int64) - 1

x_train, y_train = prepare_data(train_frame)
x_validation, y_validation = prepare_data(validation_frame)
x_train_tensor = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
x_validation_tensor = torch.tensor(x_validation, dtype=torch.float32, device=DEVICE)

log_cell("Data setup", data_started, "split=0.6/0.2/0.2, scaler=StandardScaler")


In [ ]:
LAYER_CHOICES = (1, 2, 3)
WIDTH_CHOICES = (16, 24, 32, 48, 64)
ACTIVATION_CHOICES = ("Sigmoid", "ReLU")
LEARNING_RATE_CHOICES = (0.0001, 0.001, 0.01)
BATCH_SIZE_CHOICES = (256, 512, 1024)
WEIGHT_DECAY_CHOICES = (0.0, 0.00001, 0.0001, 0.001)
GENES = {
    "hidden_layers": LAYER_CHOICES,
    "hidden_width": WIDTH_CHOICES,
    "activation": ACTIVATION_CHOICES,
    "learning_rate": LEARNING_RATE_CHOICES,
    "batch_size": BATCH_SIZE_CHOICES,
    "weight_decay": WEIGHT_DECAY_CHOICES,
}

class NASNN(nn.Module):
    def __init__(self, input_size, hidden_layers, hidden_width, activation):
        super().__init__()
        activation_layer = nn.Sigmoid if activation == "Sigmoid" else nn.ReLU
        layers = []
        previous_width = input_size
        for _ in range(hidden_layers):
            layers.extend([nn.Linear(previous_width, hidden_width), activation_layer()])
            previous_width = hidden_width
        layers.append(nn.Linear(previous_width, 5))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

def train_and_evaluate(config):
    torch.manual_seed(SEED)
    model = NASNN(
        len(feature_names),
        config["hidden_layers"],
        config["hidden_width"],
        config["activation"],
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    loss_fn = nn.CrossEntropyLoss()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    model.train()
    for _ in range(config["epochs"]):
        order = torch.randperm(len(x_train_tensor), generator=generator, device=DEVICE)
        for start in range(0, len(order), config["batch_size"]):
            batch_index = order[start : start + config["batch_size"]]
            optimizer.zero_grad()
            loss = loss_fn(model(x_train_tensor[batch_index]), y_train_tensor[batch_index])
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        prediction = model(x_validation_tensor).argmax(dim=1).cpu().numpy()
    return (
        f1_score(y_validation, prediction, average="macro"),
        balanced_accuracy_score(y_validation, prediction),
        sum(parameter.numel() for parameter in model.parameters()),
    )

def random_configuration(rng):
    return {name: values[rng.integers(len(values))] for name, values in GENES.items()}

def tournament(population, rng):
    contenders = rng.choice(population, size=2, replace=False)
    return max(contenders, key=lambda row: row["macro_f1"])

def child_configuration(parent_a, parent_b, rng, mutation_rate=0.20):
    child = {}
    for name, values in GENES.items():
        child[name] = parent_a[name] if rng.random() < 0.5 else parent_b[name]
        if rng.random() < mutation_rate:
            child[name] = values[rng.integers(len(values))]
    return child

search_started = time.perf_counter()
rng = np.random.default_rng(SEED)
population = [random_configuration(rng) for _ in range(POPULATION_SIZE)]
nas_rows = []
best_by_generation = []

for generation in range(1, GENERATIONS + 1):
    evaluated_population = []
    for configuration in population:
        trial = len(nas_rows) + 1
        config = {**configuration, "epochs": SEARCH_EPOCHS}
        print(f"Task 4 trial {trial}/{NAS_TRIALS}: generation={generation}, {config}", flush=True)
        started = time.perf_counter()
        macro_f1, balanced_accuracy, parameters = train_and_evaluate(config)
        row = {
            "trial": trial,
            "generation": generation,
            **config,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
            "parameters": parameters,
            "seconds": time.perf_counter() - started,
        }
        nas_rows.append(row)
        evaluated_population.append(row)
        print(f"Task 4 trial {trial} result: macro-F1={format_decimal(macro_f1)}, balanced accuracy={format_decimal(balanced_accuracy)}, seconds={format_decimal(row['seconds'])}", flush=True)
    evaluated_population.sort(key=lambda row: row["macro_f1"], reverse=True)
    best_by_generation.append(evaluated_population[0]["macro_f1"])
    if generation < GENERATIONS:
        population = [{name: evaluated_population[0][name] for name in GENES}]
        while len(population) < POPULATION_SIZE:
            parent_a = tournament(evaluated_population, rng)
            parent_b = tournament(evaluated_population, rng)
            population.append(child_configuration(parent_a, parent_b, rng))

nas_table = pd.DataFrame(nas_rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
best_nas = nas_table.iloc[0]
output_path = Path("/kaggle/working/task4_nas_results.csv") if Path("/kaggle").exists() else ROOT / "task4_nas_results.csv"
nas_table.to_csv(output_path, index=False)
print("Task 4 complete:")
print(nas_table.to_string(index=False, float_format=format_decimal))
print(f"Saved results: {output_path}")
log_cell("Task 4 evolutionary NAS", search_started, f"population={POPULATION_SIZE}, generations={GENERATIONS}, trials={NAS_TRIALS}, epochs_per_trial={SEARCH_EPOCHS}")


### 4.6 Best configuration, performance, and search time

The previous cell prints the best architecture, training hyperparameters, validation macro-F1, balanced accuracy, parameter count, and total runtime. Copy those measured values here after the Kaggle run.

### 4.7 Search results table

The previous cell prints every evaluated candidate ranked by validation macro-F1. Copy that measured table here after the Kaggle run.
